# Análise Comparativa do IDEB (Anos Iniciais): Paraíba vs. Rio Grande do Norte

Este notebook realiza um estudo de inferência estatística para verificar se existe diferença significativa entre as médias do **IDEB (Anos iniciais do ensino fundamental - Rede pública)** dos municípios da **Paraíba (PB)** e do **Rio Grande do Norte (RN)**, adotando um nível de significância de $\alpha = 5\%$ ($0,05$).

## Etapas da Análise
1. **Carregamento e Limpeza dos Dados:** Leitura das planilhas municipais e remoção de valores ausentes (`-`).
2. **Análise Exploratória e Visualização:** Estatísticas descritivas (média, mediana, desvio padrão, variância, mínimo e máximo) e gráficos comparativos (Boxplot e Histograma/KDE).
3. **Teste de Homogeneidade de Variâncias (Suposições):** Verificação da igualdade das variâncias populacionais via **Teste F de Snedecor** e **Teste de Levene** ($H_0: \sigma_{PB}^2 = \sigma_{RN}^2$ vs. $H_1: \sigma_{PB}^2 \neq \sigma_{RN}^2$).
4. **Teste de Hipóteses para Duas Médias:** Aplicação do **Teste $t$ de Welch** (para variâncias heterogêneas) para testar $H_0: \mu_{PB} = \mu_{RN}$ contra $H_1: \mu_{PB} \neq \mu_{RN}$.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

## 1. Carregamento e Limpeza dos Dados

In [2]:
df_pb = pd.read_excel("Dados Paraíba.xlsx")
df_rn = pd.read_excel("Dados RN.xlsx")

col_ideb = "IDEB – Anos iniciais do ensino fundamental (Rede pública)"

# Conversão para numérico (registros com '-' viram NaN e são descartados)
ideb_pb = pd.to_numeric(df_pb[col_ideb], errors="coerce").dropna()
ideb_rn = pd.to_numeric(df_rn[col_ideb], errors="coerce").dropna()

print(f"Municípios válidos na Paraíba (PB): {len(ideb_pb)} de {len(df_pb)} (6 sem nota divulgada)")
print(f"Municípios válidos no Rio Grande do Norte (RN): {len(ideb_rn)} de {len(df_rn)} (6 sem nota divulgada)")

Municípios válidos na Paraíba (PB): 217 de 223 (6 sem nota divulgada)
Municípios válidos no Rio Grande do Norte (RN): 161 de 167 (6 sem nota divulgada)

## 2. Estatísticas Descritivas e Visualização

In [3]:
resumo = pd.DataFrame({
    "Estado": ["Paraíba (PB)", "Rio Grande do Norte (RN)"],
    "N (Municípios)": [len(ideb_pb), len(ideb_rn)],
    "Média": [ideb_pb.mean(), ideb_rn.mean()],
    "Mediana": [ideb_pb.median(), ideb_rn.median()],
    "Desvio Padrão": [ideb_pb.std(ddof=1), ideb_rn.std(ddof=1)],
    "Variância": [ideb_pb.var(ddof=1), ideb_rn.var(ddof=1)],
    "Mínimo": [ideb_pb.min(), ideb_rn.min()],
    "Máximo": [ideb_pb.max(), ideb_rn.max()]
})

print(resumo.round(4).to_string(index=False))

                  Estado  N (Municípios)  Média  Mediana  Desvio Padrão  Variância  Mínimo  Máximo
            Paraíba (PB)             217 5.3438      5.2         0.8207     0.6736     3.6     8.2
Rio Grande do Norte (RN)             161 4.8379      4.8         0.6658     0.4432     3.6     7.6

In [4]:
df_plot = pd.DataFrame({
    "IDEB": pd.concat([ideb_pb, ideb_rn], ignore_index=True),
    "Estado": ["Paraíba"] * len(ideb_pb) + ["Rio Grande do Norte"] * len(ideb_rn)
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df_plot, x="Estado", y="IDEB", ax=axes[0], width=0.45, palette=["#2b5c8f", "#d95f02"])
axes[0].set_title("Distribuição do IDEB por Estado (Boxplot)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("IDEB (Anos Iniciais)")
axes[0].set_xlabel("")

sns.histplot(data=df_plot, x="IDEB", hue="Estado", kde=True, element="step", ax=axes[1], palette=["#2b5c8f", "#d95f02"])
axes[1].axvline(ideb_pb.mean(), color="#2b5c8f", linestyle="--", label=f"Média PB ({ideb_pb.mean():.2f})")
axes[1].axvline(ideb_rn.mean(), color="#d95f02", linestyle="--", label=f"Média RN ({ideb_rn.mean():.2f})")
axes[1].set_title("Histograma e Densidade do IDEB", fontsize=12, fontweight="bold")
axes[1].set_xlabel("IDEB (Anos Iniciais)")
axes[1].set_ylabel("Frequência")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Verificação das Suposições sobre as Variâncias

Antes de escolher entre o teste $t$ de Student (variâncias iguais) e o teste $t$ de Welch (variâncias diferentes), testamos a igualdade das variâncias populacionais com $\alpha = 0,05$:

* **Hipótese Nula ($H_0$):** $\sigma_{PB}^2 = \sigma_{RN}^2$ (as variâncias são homogêneas / iguais).
* **Hipótese Alternativa ($H_1$):** $\sigma_{PB}^2 \neq \sigma_{RN}^2$ (as variâncias são heterogêneas / diferentes).

In [5]:
alpha = 0.05

# 1. Teste F de Snedecor (bicaudal)
var_pb, var_rn = ideb_pb.var(ddof=1), ideb_rn.var(ddof=1)
n_pb, n_rn = len(ideb_pb), len(ideb_rn)

F_stat = var_pb / var_rn
gl_num, gl_den = n_pb - 1, n_rn - 1
p_valor_F = 2 * min(stats.f.cdf(F_stat, gl_num, gl_den), 1 - stats.f.cdf(F_stat, gl_num, gl_den))

# 2. Teste de Levene (robusto a desvios de normalidade)
W_stat, p_valor_levene = stats.levene(ideb_pb, ideb_rn, center="median")

print(f"Teste F: Estatística F = {F_stat:.4f} | gl = ({gl_num}, {gl_den}) | p-valor = {p_valor_F:.4f}")
print(f"Teste de Levene: Estatística W = {W_stat:.4f} | p-valor = {p_valor_levene:.4f}")

if p_valor_F < alpha and p_valor_levene < alpha:
    print("\nConclusão sobre variâncias: Rejeita-se H0 (p < 0,05). As variâncias são significativamente DIFERENTES.")
    print("Procedimento indicado: Teste t de Welch (equal_var=False).")
else:
    print("\nConclusão sobre variâncias: Não se rejeita H0. Assume-se variâncias IGUAIS (Teste t de Student).")

Teste F: Estatística F = 1.5197 | gl = (216, 160) | p-valor = 0.0053
Teste de Levene: Estatística W = 5.6143 | p-valor = 0.0183

Conclusão sobre variâncias: Rejeita-se H0 (p < 0,05). As variâncias são significativamente DIFERENTES.
Procedimento indicado: Teste t de Welch (equal_var=False).

## 4. Teste de Hipóteses para Comparação das Médias

Comprovada a **heterogeneidade das variâncias** ($s_{PB}^2 = 0,6736$ vs. $s_{RN}^2 = 0,4432$; $p = 0,0053$ no Teste F e $p = 0,0183$ no Teste de Levene), aplicamos o **Teste $t$ de Welch** para amostras independentes com variâncias desiguais:

* **Hipótese Nula ($H_0$):** $\mu_{PB} = \mu_{RN}$ (não há diferença significativa entre as médias do IDEB dos municípios da PB e do RN).
* **Hipótese Alternativa ($H_1$):** $\mu_{PB} \neq \mu_{RN}$ (há diferença significativa entre as médias do IDEB dos municípios da PB e do RN).

In [6]:
t_stat, p_valor_t = stats.ttest_ind(ideb_pb, ideb_rn, equal_var=False)

# Graus de liberdade aproximados de Welch-Satterthwaite
s1_n1 = var_pb / n_pb
s2_n2 = var_rn / n_rn
gl_welch = (s1_n1 + s2_n2)**2 / ((s1_n1**2) / (n_pb - 1) + (s2_n2**2) / (n_rn - 1))

# Intervalo de Confiança de 95% para a diferença das médias (PB - RN)
diff_medias = ideb_pb.mean() - ideb_rn.mean()
erro_padrao_diff = np.sqrt(s1_n1 + s2_n2)
ic_95 = stats.t.interval(0.95, df=gl_welch, loc=diff_medias, scale=erro_padrao_diff)

print(f"Estatística t (Welch): {t_stat:.4f}")
print(f"Graus de liberdade (Welch): {gl_welch:.2f}")
print(f"p-valor (bicaudal): {p_valor_t:.6e}")
print(f"Diferença entre as médias (PB - RN): {diff_medias:.4f}")
print(f"IC 95% para a diferença: [{ic_95[0]:.4f}, {ic_95[1]:.4f}]")

Estatística t (Welch): 6.6102
Graus de liberdade (Welch): 372.98
p-valor (bicaudal): 1.329834e-10
Diferença entre as médias (PB - RN): 0.5059
IC 95% para a diferença: [0.3554, 0.6564]

## 5. Interpretação dos Resultados

1. **Suposição das Variâncias:** Tanto pelo Teste F ($F = 1,5197; p = 0,0053$) quanto pelo Teste de Levene ($W = 5,6143; p = 0,0183$), rejeitamos a hipótese de igualdade das variâncias ao nível de 5%. Os municípios da Paraíba apresentam maior dispersão nas notas do IDEB ($s^2 = 0,6736$) em relação aos municípios do Rio Grande do Norte ($s^2 = 0,4432$). Por isso, o **Teste $t$ de Welch** é a escolha estatisticamente adequada.
2. **Comparação das Médias:** O Teste $t$ de Welch resultou em $t = 6,6102$ com $p\text{-valor} = 1,35 \times 10^{-10}$ ($p < 0,05$). Portanto, **rejeitamos a hipótese nula ($H_0$)**.
3. **Conclusão Prática:** Há evidência estatística altamente significativa de que as médias do IDEB (anos iniciais da rede pública) diferem entre os dois estados. A Paraíba apresenta média superior ($5,34$) à do Rio Grande do Norte ($4,84$), com uma diferença média estimada de **$+0,51$ pontos** (IC 95%: $[0,36; 0,66]$).